# BIOM 200: Microbiome analysis with QIIME 2

Instructors: Rob Knight (robknight@ucsd.edu), Lucas Patel (lpatel@health.ucsd.edu)

The goal is to familiarize yourself with and analyze 16S rRNA amplicon data from Qiita, starting with a feature table and ending with differential abundance. Run the cells in order. Edit only the cells marked **EDIT**. The analysis and plotting functions live in `scripts/biom200.py`; each plot has controls for changing what it shows, and every plot is saved as a PDF in `figures/`. The first part analyzes the class study; the final section, *Your own samples*, repeats the analysis on your group's samples.

## Introduction

The microbes living on and in us shape, and are shaped by, our immune system, our metabolism, and our susceptibility to pathogens. Different body sites host very different communities, and those communities vary far more between people than within a person over time. 16S rRNA amplicon sequencing is the most widely used sequencing modality to ask which bacteria are present in a sample and in what proportions: it is relatively cheap, fast enough to process within a day, and supported by mature, time-tested software. It also has well-known limits. It measures relative rather than absolute abundance, it rarely resolves species, and small studies are easily misled by contamination, sequencing depth, and multiple testing burden. This notebook walks through a standard analysis with those limits in view.

**Read more:** [Gilbert et al. 2018, *Nat Med*](https://doi.org/10.1038/nm.4517) · [Costello et al. 2009, *Science*](https://doi.org/10.1126/science.1177486) · [Knight et al. 2018, *Nat Rev Microbiol*](https://doi.org/10.1038/s41579-018-0029-9)

## Configuring QIIME 2

QIIME 2 is an open-source platform for microbiome analysis. This notebook uses the **QIIME 2 2026.7** distribution. Choose one option.

**Read more:** [Bolyen et al. 2019, *Nat Biotechnol*](https://doi.org/10.1038/s41587-019-0209-9)

### Option A (recommended): GitHub Codespaces

Nothing to install. On the repository page, click **Code → Codespaces → Create codespace on main**. The first start takes a few minutes; QIIME 2 and this notebook open in your browser. When asked for a kernel, choose **QIIME 2 2026.7**. Stop the codespace when you finish (**Codespaces** menu → **Stop Current Codespace**) to save your free monthly hours.

### Option B: install on your own computer

Full instructions: [QIIME 2 installation guide](https://library.qiime2.org/quickstart/amplicon).

1. Install [Miniconda](https://www.anaconda.com/docs/getting-started/miniconda/install). **Windows:** first [install WSL](https://learn.microsoft.com/en-us/windows/wsl/install), then install Miniconda *inside* WSL and follow the Linux instructions.
2. Create the environment for your system (takes 10–30 minutes):

**Linux or Windows (WSL)**
```bash
conda env create --name rachis-qiime2-2026.7 \
  --file https://raw.githubusercontent.com/qiime2/distributions/refs/heads/dev/2026.7/qiime2/released/rachis-qiime2-linux-64-conda.yml
```

**macOS, Apple Silicon (M1 and later)**: runs under Rosetta 2
```bash
CONDA_SUBDIR=osx-64 conda env create --name rachis-qiime2-2026.7 \
  --file https://raw.githubusercontent.com/qiime2/distributions/refs/heads/dev/2026.7/qiime2/released/rachis-qiime2-osx-64-conda.yml
conda activate rachis-qiime2-2026.7
conda config --env --set subdir osx-64
```

**macOS, Intel**
```bash
conda env create --name rachis-qiime2-2026.7 \
  --file https://raw.githubusercontent.com/qiime2/distributions/refs/heads/dev/2026.7/qiime2/released/rachis-qiime2-osx-64-conda.yml
```

3. Add Jupyter (QIIME 2 2026.7 no longer bundles it) and open this notebook:
```bash
conda activate rachis-qiime2-2026.7
conda install -c conda-forge notebook
jupyter notebook biom200_microbiome.ipynb
```

The next cell confirms that the notebook is running inside the QIIME 2 environment.

In [1]:
%matplotlib inline
import qiime2
print("QIIME 2", qiime2.__version__)

QIIME 2 2026.7.0


## Loading data from Qiita

Qiita is a web platform that stores microbiome studies together with their metadata and processes every study the same way, so results can be compared across studies. Our reads were amplified from the 16S V4 region with the Earth Microbiome Project protocol, denoised into exact amplicon sequence variants (ASVs) with Deblur, and placed on the Greengenes2 reference tree, which provides both taxonomy and phylogeny.

**Read more:** [Gonzalez et al. 2018, *Nat Methods*](https://doi.org/10.1038/s41592-018-0141-9) · [Thompson et al. 2017, *Nature*](https://doi.org/10.1038/nature24621) · [Amir et al. 2017, *mSystems*](https://doi.org/10.1128/mSystems.00191-16) · [McDonald et al. 2024, *Nat Biotechnol*](https://doi.org/10.1038/s41587-023-01845-1)

For this class exercise, the data will be processed automatically to focus on secondary analysis. To find the input files, open a study on Qiita, and locate each file. Download input files from Qiita to your computer, then put them in the `data/` folder next to this notebook. **In a Codespace**, drag the files onto the `data` folder in the file explorer on the left, or right-click `data` → **Upload...**

| File | What it is | Where to find it in Qiita |
|---|---|---|
| Feature table (`.biom` or `.qza`) | Read counts, samples × ASVs | Study page → data type (16S) → preparation → Deblur feature table → **Download** |
| Sample information (`.tsv`) | One row per sample: body site, participant, sample type, questionnaire | Study page → **Sample Information** → Download |
| Taxonomy (`.qza` or `.tsv`) | Greengenes2 name for each ASV | Publicly available from [Greengenes2](https://ftp.microbio.me/greengenes_release/current/) (`2024.09.taxonomy.asv.tsv.qza`) |
| Phylogeny (`.qza` or `.nwk`) | Pre-constructed phylogenetic tree for UniFrac analyses | Publicly available from [Greengenes2](https://ftp.microbio.me/greengenes_release/current/) (`2024.09.phylogeny.asv.nwk.qza`) |

The repository includes a worked example so that every cell runs before you have your own data: the *Moving Pictures of the Human Microbiome* study (Qiita study 550; gut, tongue, and both palms of two people sampled over time), with Greengenes2 taxonomy and phylogeny pruned to its ASVs. See `data/README.md`.

In [ ]:
# EDIT: paths to the study files. The defaults are the bundled example, Qiita study 550 (Moving Pictures).
TABLE = "data/mp/550_feature-table.biom"       # .biom or .qza
METADATA = "data/mp/550_sample_information.tsv"
TAXONOMY = "data/mp/550_taxonomy.tsv"          # .qza or .tsv; set to None if the table carries taxonomy
TREE = "data/mp/550_tree.nwk"                  # optional rooted tree (.qza or .nwk) for Faith's PD and UniFrac

# EDIT: metadata columns
GROUP = "common_sample_site"                # the main comparison of interest
SUBJECT = "host_subject_id"                 # participant identifier
CONTROL_COL = "sample_type"                 # column that marks blanks and mock communities
CONTROL_VALUES = ["control blank", "control mock"]

In [ ]:
# Setup: the analysis and plotting functions live in scripts/biom200.py, shared by both notebooks.
# Open that file to see how each step works.
from scripts.biom200 import *

In [ ]:
study1 = load_dataset("study1", TABLE, METADATA, TAXONOMY, TREE, group=GROUP, subject=SUBJECT,
                      control_col=CONTROL_COL, control_values=CONTROL_VALUES)

### Handling controls

Extraction kits and reagents carry their own trace bacterial DNA, and DNA can leak between neighbouring wells on a plate. In low-biomass samples such as nasal swabs, this contamination can make up a large share of the reads. Blanks reveal what contamination looks like; a mock community of known composition shows whether the pipeline recovers the truth. The cell above listed the most abundant taxa in each control and removed the controls from the dataset used below. Do the blanks look like any of your samples?

**Read more:** [Salter et al. 2014, *BMC Biol*](https://doi.org/10.1186/s12915-014-0087-z) · [Minich et al. 2019, *mSystems*](https://doi.org/10.1128/mSystems.00186-19)

## Taxonomic composition

A stacked bar plot of relative abundance is often the first look at any microbiome dataset: it shows which taxa dominate each sample and whether samples from the same body site resemble each other. Because each bar sums to one, the plot shows proportions only. A taxon whose share rises may have grown, or every other taxon may have shrunk; the data cannot tell these apart.

**Read more:** [Caporaso et al. 2011, *Genome Biol*](https://doi.org/10.1186/gb-2011-12-5-r50) · [Costello et al. 2009, *Science*](https://doi.org/10.1126/science.1177486) · [Gloor et al. 2017, *Front Microbiol*](https://doi.org/10.3389/fmicb.2017.02224)

Change the taxonomic level, the field used to group samples, and the number of taxa shown; tick **Group means** to average samples within each group. Every plot is saved as a PDF in `figures/`. For the interactive QIIME 2 bar plot, run `add_composition(study1, save_qzv=True)` and open the `.qzv` file in `results/` at [view.qiime2.org](https://view.qiime2.org).

In [ ]:
add_composition(study1)
explore_composition(study1)

## Rarefaction

Samples are sequenced to different depths, and deeper samples appear more diverse simply because more reads were drawn. Rarefaction subsamples every sample, without replacement, to a common depth so that diversity is compared on equal terms. It discards data, and whether that is acceptable has been debated; for diversity analyses with very uneven depths it remains a robust choice.

**Read more:** [Weiss et al. 2017, *Microbiome*](https://doi.org/10.1186/s40168-017-0237-y) · [McMurdie & Holmes 2014, *PLoS Comput Biol*](https://doi.org/10.1371/journal.pcbi.1003531)

Move the slider to a depth where most curves have flattened, and note how many samples fall below it and would be dropped. The value left on the slider is used in the next section.

In [ ]:
add_curves(study1)
explore_rarefaction(study1)

## Alpha and beta diversity

Alpha diversity summarizes a single sample: how many taxa it contains (richness), how evenly reads are spread among them (Shannon entropy), or how much of the tree of life they span (Faith's phylogenetic diversity). Beta diversity measures how different two samples are. Bray–Curtis and Jaccard count shared taxa; UniFrac, developed in the Knight lab, also weighs how closely related the unshared taxa are phylogenetically, so two samples dominated by different *Streptococcus* species count as similar.

**Read more:** [Faith 1992, *Biol Conserv*](https://doi.org/10.1016/0006-3207(92)91201-3) · [Lozupone & Knight 2005, *Appl Environ Microbiol*](https://doi.org/10.1128/AEM.71.12.8228-8235.2005) · [Lozupone et al. 2011, *ISME J*](https://doi.org/10.1038/ismej.2010.133)

The next cell rarefies to the depth chosen above and computes the standard metrics. Rerun it after moving the slider. Faith's PD and UniFrac are included when a phylogenetic tree is provided.

In [ ]:
add_diversity(study1)

### Alpha diversity

The test here treats each sample as independent (Mann–Whitney for two groups, Kruskal–Wallis for more). If each participant contributes one sample per group, a paired test such as `scipy.stats.wilcoxon` is more appropriate.

In [ ]:
explore_alpha(study1)

### Beta diversity

Every cell is the distance between two samples, ordered by group; lighter means more similar. Light blocks along the diagonal mean that samples within a group resemble each other. The title compares the mean distance within groups with the mean distance between groups; for technical replicates, the within-group distance is the noise floor.

In [ ]:
explore_distance_matrix(study1)

## Ordination (PCoA) and PERMANOVA

A distance matrix with dozens of samples is hard to read directly. Principal coordinates analysis (PCoA) projects it into a few axes that capture as much of the variation as possible; each axis label gives the share of variation it explains. PERMANOVA then asks whether samples in different groups are more different from each other than samples within a group, using permutations rather than distributional assumptions. It is also sensitive to differences in spread between groups, which `skbio.stats.distance.permdisp` tests. For EMPeror, the Knight lab's interactive 3D viewer, run `add_diversity(study1, save_qzv=True)` to save one `.qzv` per metric to `results/`.

**Read more:** [Anderson 2001, *Austral Ecol*](https://doi.org/10.1111/j.1442-9993.2001.01070.pp.x) · [Vázquez-Baeza et al. 2013, *GigaScience*](https://doi.org/10.1186/2047-217X-2-16)

In [4]:
explore_pcoa(study1)

NameError: name 'study1' is not defined

PERMANOVA for every metric, by the main comparison and by participant:

In [ ]:
permanova_table(study1)

## Differential abundance

Which taxa differ between groups? Because 16S data are compositional, an increase in one taxon makes every other taxon look smaller, so ordinary per-taxon tests on relative abundances can report changes that never happened. Methods such as ANCOM-BC estimate and correct for this bias, and reference frames make clear that only *ratios* between taxa are truly measured. Even so, different methods applied to the same data disagree substantially, so treat any result from a small study as a hypothesis rather than a finding.

**Read more:** [Morton et al. 2019, *Nat Commun*](https://doi.org/10.1038/s41467-019-10656-5) · [Lin & Peddada 2020, *Nat Commun*](https://doi.org/10.1038/s41467-020-17041-7) · [Nearing et al. 2022, *Nat Commun*](https://doi.org/10.1038/s41467-022-28034-z)

Genera present in fewer than `min_prevalence` of samples are filtered out first, since they cannot be tested meaningfully. `reference` sets the group every other group is compared with (default: first alphabetically).

In [ ]:
add_differential_abundance(study1, min_prevalence=0.25, reference=None)
explore_forest(study1)

Questions to consider:

- How many genera were tested, and is the reported q-value corrected for that number?
- Is the genus abundant enough in the bar plot to be a plausible biological difference, or is it rare?
- Does it also appear in the blanks?

## Your own samples

Now repeat the analysis on the samples designed and collected by the group. Every step reuses the functions defined above, so each cell is a single line. Download your group's feature table and sample information from Qiita, upload them to `data/`, and **EDIT** the cell below. Set `MY_GROUP` to the metadata field that encodes your comparison (for example, before vs. after brushing).

Before interpreting anything, check two things: that your blanks are low and unlike your samples, and that each group has enough samples left after rarefaction. With a handful of samples per group, treat every p-value as a description of these samples, not of people in general.

In [ ]:
# EDIT: your group's files and comparison. The defaults are a stand-in: left vs. right palm from the example study.
MY_TABLE = "data/mp/550_feature-table.biom"
MY_METADATA = "data/mp/550_palms_sample_information.tsv"
MY_TAXONOMY = TAXONOMY                      # same Greengenes2 taxonomy as above, or None if the table carries it
MY_TREE = TREE
MY_GROUP = "common_sample_site"             # the field that encodes your comparison
MY_SUBJECT = SUBJECT

mine = load_dataset("my_samples", MY_TABLE, MY_METADATA, MY_TAXONOMY, MY_TREE, group=MY_GROUP, subject=MY_SUBJECT,
                    control_col=CONTROL_COL, control_values=CONTROL_VALUES)

### Composition

In [ ]:
add_composition(mine)
explore_composition(mine)

### Rarefaction

Choose a depth, then run the next cell.

In [ ]:
add_curves(mine)
explore_rarefaction(mine)

### Alpha and beta diversity

In [ ]:
add_diversity(mine)

In [ ]:
explore_alpha(mine)

In [ ]:
explore_distance_matrix(mine)

In [ ]:
explore_pcoa(mine)

In [ ]:
permanova_table(mine)

### Differential abundance

In [ ]:
add_differential_abundance(mine, min_prevalence=0.25, reference=None)
explore_forest(mine)

Compare your results with your pre-registered prediction. Which effects are larger than the differences between technical replicates, and which would you defend in a paper?